# Create model for prediction of gamma shower

## Load modules and data

In [ ]:
# import modules
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, validation_curve
from sklearn.impute import SimpleImputer
import pickle
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

from imblearn.pipeline import Pipeline 
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve

import optuna 
from optuna.samplers import TPESampler
import time

c:\Users\moonp\Documents\Beruflich\Weiterbildung\Stackfuel - Data Science\Portfolio Projekt\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# show three digits
pd.set_option('display.float_format', '{:,.3f}'.format)
pd.set_option('display.precision', 3)
np.set_printoptions(precision=3)

In [11]:
# perform train-test-split
columns = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
    "class"
]

df = pd.read_csv(
    r"../data/raw/magic04.data",
    header=None,
    names=columns
)

# remove dublicates before train-test-split
df = df.drop_duplicates().reset_index(drop=True)

features = df.iloc[:, :-1]
target = df.iloc[:, -1]

features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.1, random_state=42)

In [14]:
# save features_test as 'features_test.csv'
features_train.to_csv('../data/rawfeatures_train_unprocessed.csv', index=False)

In [15]:
# save features_test as 'features_test.csv'
features_test.to_csv('../data/rawfeatures_test_unprocessed.csv', index=False)

## Data Preparation

### Datatype transformation

Die Daten liegen bereits im richtigen Format vor und müssen nicht umgewandelt werden

### Data Imputation

Der Datensatz beinhaltet keine fehlenden Werte, es müssen also keine Daten imputiert werden

### Resampling

Da die Zielklassen mit etwa 65% (g) zu 35% (h) leicht unausgegleichen sind, wird über eine Kreuzvalidierung mit GridSearch ermittelt, welches Verfahren für diesen Datensatz am geeignetsten für ein Resampling ist

In [ ]:
model_log = LogisticRegression(solver='lbfgs', random_state=42)
search_space = [{'estimator__C': [0.001, 0.01, 0.1, 1, 10, 100], 'estimator__class_weight': [None, 'balanced']}]
samplers = [('oversampling', RandomOverSampler(random_state=42)),
            ('undersampling', RandomUnderSampler(random_state=42)),
            ('class_weights', 'passthrough'),
            ('SMOTE', SMOTE(random_state = 42))
           ]
results = []
for name, sampler in samplers:
    imb_pipeline = Pipeline([('sampler', sampler), ('estimator', model_log)])
    grid_search = GridSearchCV(estimator = imb_pipeline,
                              param_grid = search_space,
                               cv = 5,
                              scoring = 'roc_curve',
                              n_jobs = -1)
    grid_search.fit(features_train.loc[:,['fLength', 'fWidth', 'fAlpha'] ], target_train)
    model = grid_search.best_estimator_.named_steps['estimator']
    print(name)
    #print(grid_search.scoring, 'on Validationset:', grid_search.best_score_ )
    print(f"{grid_search.scoring} on Validationset: {grid_search.best_score_:.3f}")
    print(model)
    print('#'*11)

InvalidParameterError: The 'scoring' parameter of GridSearchCV must be a str among {'neg_mean_absolute_error', 'recall', 'd2_absolute_error_score', 'homogeneity_score', 'roc_auc', 'accuracy', 'neg_max_error', 'positive_likelihood_ratio', 'roc_auc_ovo_weighted', 'neg_mean_absolute_percentage_error', 'recall_weighted', 'matthews_corrcoef', 'v_measure_score', 'completeness_score', 'jaccard_micro', 'f1_micro', 'f1_macro', 'jaccard_weighted', 'neg_median_absolute_error', 'neg_mean_squared_log_error', 'top_k_accuracy', 'neg_brier_score', 'precision_weighted', 'fowlkes_mallows_score', 'precision_macro', 'neg_root_mean_squared_error', 'mutual_info_score', 'roc_auc_ovr_weighted', 'f1', 'recall_macro', 'adjusted_rand_score', 'r2', 'neg_mean_poisson_deviance', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_log_error', 'precision', 'd2_brier_score', 'f1_weighted', 'jaccard_macro', 'jaccard_samples', 'balanced_accuracy', 'recall_micro', 'precision_samples', 'explained_variance', 'neg_log_l

: 